In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install mne pyedflib numpy scipy

In [ ]:
import mne
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.signal import welch
from scipy.stats import entropy
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim

# --- PARAMETERS ---
edf_folder = '/content/drive/MyDrive/BME473PoP ML/Seizure edfs'
summary_folder = "/content/drive/MyDrive/BME473PoP ML/Summaries"
sfreq = 256

PREICTAL_DURATION = 5 * 60  # 5 minute preictal window

window_size_s = 5  # seconds
window_size = window_size_s * sfreq
common_channels = None
def parse_summary(subject_id):
    summary_path = os.path.join(summary_folder, f"{subject_id}-summary.txt")
    intervals = {}

    if not os.path.exists(summary_path):
        print("WARNING: Missing summary for", subject_id)
        return intervals

    with open(summary_path, "r") as f:
        lines = f.readlines()

    current_file = None
    start = None

    for line in lines:
        line = line.strip()

        if line.startswith("File Name:"):
            current_file = line.split(":")[1].strip()
            intervals[current_file] = []
            start = None
            continue

        # Handle any "Seizure X Start Time" / "Seizure X End Time"
        if "Seizure" in line and "Start Time" in line:
            value = line.split(":")[1].replace("seconds", "").strip()
            start = float(value)

        if "Seizure" in line and "End Time" in line and start is not None:
            value = line.split(":")[1].replace("seconds", "").strip()
            end = float(value)
            intervals[current_file].append((start, end))
            start = None

    return intervals

In [ ]:
# --- TRAIN / TEST SPLIT BY EDF ---
edf_files = [f for f in os.listdir(edf_folder) if f.endswith(".edf")]
edf_files = sorted(edf_files)

train_files, test_files = train_test_split(
    edf_files,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Train EDFs:", len(train_files))
print("Test EDFs:", len(test_files))


In [ ]:
# ===================== LOAD PREPROCESSED DATA =====================

LOAD_PATH = "/content/drive/MyDrive/BME473PoP ML/windowed_dataset_one_channel.npz"

# Load dataset
data = np.load(LOAD_PATH)

train_segments = data["X_train"]
train_labels   = data["y_train"]

test_segments  = data["X_test"]
test_labels    = data["y_test"]

print("Loaded preprocessed dataset")
print("Train shape:", train_segments.shape, train_labels.shape)
print("Test shape:", test_segments.shape, test_labels.shape)


In [ ]:
# Window one channel (T7-P7) of data from CHBMIT
# No need to run if loading prewindowed data
train_segments = []
train_labels = []

test_segments = []
test_labels = []

standard_channels = ["T7-P7"]

for edf in edf_files:
    full_path = os.path.join(edf_folder, edf)
    subject_id = edf[:5]
    intervals_dict = parse_summary(subject_id)
    seizure_intervals = intervals_dict.get(edf, [])

    if len(seizure_intervals) == 0:
        print(f"Skipping {edf} (no seizures)")
        continue

    print(f"Processing {edf} | Seizures: {seizure_intervals}")

    raw = mne.io.read_raw_edf(full_path, preload=True, verbose=False)
    raw.filter(0.5, 40, verbose=False)
    raw.notch_filter(60, verbose=False)

    data = raw.get_data()

    # enforce channel set
    ch_idx = []
    for ch in standard_channels:
        matches = [i for i, name in enumerate(raw.ch_names) if name.startswith(ch)]
        if matches:
            ch_idx.append(matches[0])
        else:
            print(f"Skipping {edf} (missing channel {ch})")
            ch_idx = []
            break

    if len(ch_idx) != len(standard_channels):
        continue

    data = data[ch_idx, :]
    n_samples = data.shape[1]

    segments = []
    labels = []

    for start_sample in range(0, n_samples - window_size + 1, window_size):
        end_sample = start_sample + window_size
        t_start = start_sample / sfreq
        t_end = end_sample / sfreq

        label = 0  # interictal default

        for (sz_start, sz_end) in seizure_intervals:

            # ictal overlap
            ictal_overlap = max(0, min(t_end, sz_end) - max(t_start, sz_start))
            if ictal_overlap / window_size_s >= 0.5:
                label = 2
                break

            # pre-ictal
            pre_start = max(0, sz_start - PREICTAL_DURATION)
            pre_end = sz_start

            pre_overlap = max(0, min(t_end, pre_end) - max(t_start, pre_start))
            if pre_overlap / window_size_s >= 0.5:
                label = max(label, 1)
        #reduced data size so maybe if bad performance to back to float 64 with colab pro
        seg_float32 = data[:, start_sample:end_sample].astype(np.float32)
        segments.append(seg_float32)
        labels.append(label)

    segments = np.array(segments)
    labels = np.array(labels)

    ictal = segments[labels == 2]
    preictal = segments[labels == 1]
    interictal = segments[labels == 0]

    if len(preictal) == 0 or len(ictal) == 0:
        continue

    #this cap might not be stringent enough because class imbalance is still
    # a large issue. 1.5x might be better
    cap = int(1.25 * len(preictal))

    if len(interictal) > cap:
        interictal = resample(interictal, replace=False,
                              n_samples=cap, random_state=42)

    segments_balanced = np.concatenate([ictal, preictal, interictal])
    labels_balanced = np.concatenate([
        np.full(len(ictal), 2),
        np.full(len(preictal), 1),
        np.full(len(interictal), 0)
    ])

    if edf in train_files:
        for seg, lbl in zip(segments_balanced, labels_balanced):
            train_segments.append(seg)
            train_labels.append(lbl)
    else:
        for seg, lbl in zip(segments_balanced, labels_balanced):
            test_segments.append(seg)
            test_labels.append(lbl)

In [ ]:
# ===================== SAVE PREPROCESSED DATA =====================
# No need to run if loading prewindowed data
import os

# Directory to store saved data
SAVE_DIR = "/content/drive/MyDrive/BME473PoP ML"
os.makedirs(SAVE_DIR, exist_ok=True)

SAVE_PATH = os.path.join(SAVE_DIR, "windowed_dataset_one_channel.npz")

# Convert lists to arrays (important for reload consistency)
train_segments = np.array(train_segments, dtype=np.float32)
train_labels   = np.array(train_labels, dtype=np.int64)

test_segments  = np.array(test_segments, dtype=np.float32)
test_labels    = np.array(test_labels, dtype=np.int64)

# Save compressed dataset
np.savez_compressed(
    SAVE_PATH,
    X_train=train_segments,
    y_train=train_labels,
    X_test=test_segments,
    y_test=test_labels
)

print("Saved preprocessed dataset to:", SAVE_PATH)
print("Train shape:", train_segments.shape, train_labels.shape)
print("Test shape:", test_segments.shape, test_labels.shape)


Saved preprocessed dataset to: /content/drive/MyDrive/BME473PoP ML/windowed_dataset_one_channel.npz
Train shape: (20917, 1, 1280) (20917,)
Test shape: (5145, 1, 1280) (5145,)


In [ ]:
X_train = np.array(train_segments)
y_train = np.array(train_labels)

X_test = np.array(test_segments)
y_test = np.array(test_labels)

print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

print("Train class counts:", np.bincount(y_train))
print("Test class counts:", np.bincount(y_test))

In [ ]:
X_train = np.array(train_segments)
y_train = np.array(train_labels)
X_test = np.array(test_segments)
y_test = np.array(test_labels)

print("Before balancing:")
print("Train class counts:", np.bincount(y_train))
print("Test class counts:", np.bincount(y_test))

# --- GLOBAL REBALANCING ---
# Oversample ictal and preictal to match interictal, or
# undersample everything to match the minority class (ictal)
from sklearn.utils import resample

def balance_dataset(X, y):
    classes = np.unique(y)
    # Find minority class count
    min_count = min(np.bincount(y))
    X_resampled, y_resampled = [], []
    for c in classes:
        X_c = X[y == c]
        y_c = y[y == c]
        if len(X_c) > min_count:
            # Undersample majority classes
            X_c, y_c = resample(X_c, y_c, replace=False,
                                n_samples=min_count, random_state=42)
        else:
            # Oversample minority classes
            X_c, y_c = resample(X_c, y_c, replace=True,
                                n_samples=min_count, random_state=42)
        X_resampled.append(X_c)
        y_resampled.append(y_c)
    return np.concatenate(X_resampled), np.concatenate(y_resampled)

X_train, y_train = balance_dataset(X_train, y_train)
X_test, y_test = balance_dataset(X_test, y_test)

print("After balancing:")
print("Train class counts:", np.bincount(y_train))
print("Test class counts:", np.bincount(y_test))

Before balancing:
Train class counts: [10601  8485  1831]
Test class counts: [2632 2121  392]
After balancing:
Train class counts: [1831 1831 1831]
Test class counts: [392 392 392]


In [ ]:
print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)
print("Train class counts:", np.bincount(y_train))
print("Test class counts:", np.bincount(y_test))

# Per-sample normalization — each window normalized independently
# More robust than global when amplitude varies across patients/files
def normalize_per_sample(X):
    mean = X.mean(axis=-1, keepdims=True)
    std = X.std(axis=-1, keepdims=True) + 1e-8
    return (X - mean) / std

X_train = normalize_per_sample(X_train)
X_test = normalize_per_sample(X_test)

class EEGDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = EEGDataset(X_train, y_train)
test_dataset = EEGDataset(X_test, y_test)

Train set: (5493, 1, 1280) (5493,)
Test set: (1176, 1, 1280) (1176,)
Train class counts: [1831 1831 1831]
Test class counts: [392 392 392]


In [ ]:
batch_size = 64

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recompute class weights after balancing
class_counts = np.bincount(y_train)
class_weights = 1.0 / np.sqrt(class_counts)
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights_tensor)

# Weighted cross entropy — simpler and less gameable than focal+margin
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=0.1
)

In [ ]:
from scipy.signal import welch

def compute_bandpower_features(X_raw, sfreq=256):
    """
    X_raw: numpy array of shape (n_samples, 1, n_timepoints)
    Returns: (n_samples, n_features) array of bandpower features
    """
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta':  (13, 30),
        'gamma': (30, 40),
    }
    features = []
    for i in range(X_raw.shape[0]):
        sig = X_raw[i, 0, :]  # shape: (n_timepoints,)
        freqs, psd = welch(sig, fs=sfreq, nperseg=sfreq)
        row = []
        total_power = np.trapz(psd, freqs) + 1e-8
        for band, (lo, hi) in bands.items():
            idx = np.logical_and(freqs >= lo, freqs <= hi)
            bp = np.trapz(psd[idx], freqs[idx])
            row.append(bp / total_power)  # relative power
        # Also add log variance of raw signal
        row.append(np.log(np.var(sig) + 1e-8))
        features.append(row)
    return np.array(features, dtype=np.float32)

print("Computing bandpower features (this may take a minute)...")
X_train_bp = compute_bandpower_features(X_train)
X_test_bp  = compute_bandpower_features(X_test)

# Normalize bandpower features
bp_mean = X_train_bp.mean(axis=0)
bp_std  = X_train_bp.std(axis=0) + 1e-8
X_train_bp = (X_train_bp - bp_mean) / bp_std
X_test_bp  = (X_test_bp  - bp_mean) / bp_std

print("Bandpower feature shape:", X_train_bp.shape)


class EEGDatasetDual(torch.utils.data.Dataset):
    def __init__(self, X_raw, X_bp, y):
        self.X_raw = torch.tensor(X_raw, dtype=torch.float32)
        self.X_bp  = torch.tensor(X_bp,  dtype=torch.float32)
        self.y     = torch.tensor(y,      dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_raw[idx], self.X_bp[idx], self.y[idx]


train_dataset = EEGDatasetDual(X_train, X_train_bp, y_train)
test_dataset  = EEGDatasetDual(X_test,  X_test_bp,  y_test)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(
    test_dataset,  batch_size=64, shuffle=False)


class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.bn1   = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.bn2   = nn.BatchNorm1d(channels)

    def forward(self, x):
        residual = x
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return torch.relu(x + residual)


class EEGNetDual(nn.Module):
    def __init__(self, n_bp_features):
        super().__init__()
        # --- Raw signal branch (CNN) ---
        self.conv1 = nn.Conv1d(1, 32, kernel_size=25, padding=12)
        self.bn1   = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(4)

        self.res1  = ResBlock(32, kernel_size=7)
        self.pool2 = nn.MaxPool1d(4)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2   = nn.BatchNorm1d(64)
        self.pool3 = nn.MaxPool1d(4)

        self.res2      = ResBlock(64, kernel_size=3)
        self.global_avg = nn.AdaptiveAvgPool1d(1)

        # --- Bandpower branch (MLP) ---
        self.bp_fc1 = nn.Linear(n_bp_features, 32)
        self.bp_bn1 = nn.BatchNorm1d(32)
        self.bp_fc2 = nn.Linear(32, 32)

        # --- Fusion head ---
        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(64 + 32, 3)  # CNN out + BP out

    def forward(self, x_raw, x_bp):
        # CNN branch
        x = torch.relu(self.bn1(self.conv1(x_raw)))
        x = self.pool1(x)
        x = self.res1(x)
        x = self.pool2(x)
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.pool3(x)
        x = self.res2(x)
        x = self.global_avg(x).squeeze(-1)  # (batch, 64)

        # Bandpower branch
        b = torch.relu(self.bp_bn1(self.bp_fc1(x_bp)))
        b = torch.relu(self.bp_fc2(b))        # (batch, 32)

        # Fuse and classify
        out = torch.cat([x, b], dim=1)        # (batch, 96)
        out = self.dropout(out)
        return self.fc(out)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_bp_features = X_train_bp.shape[1]
model = EEGNetDual(n_bp_features=n_bp_features).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3)
print(f"Model built. BP features: {n_bp_features}")

In [ ]:
interictal_indices = np.where(y_train == 0)[0]
sample = X_train[interictal_indices[0], 0, :]  # shape: (1280,)
print("Shape:", sample[:200].shape)
print(sample[:200])

Shape: (200,)
[ 4.82187092e-01  4.83682364e-01  4.54654634e-01  5.00399888e-01
  6.38938129e-01  8.06346416e-01  9.26714838e-01  9.80081379e-01
  1.00839663e+00  1.06351185e+00  1.15517306e+00  1.25183332e+00
  1.32966053e+00  1.41340101e+00  1.55854762e+00  1.78589869e+00
  2.03388476e+00  2.18600965e+00  2.16193008e+00  1.99471307e+00
  1.82166588e+00  1.78882420e+00  1.94602215e+00  2.21783113e+00
  2.46793151e+00  2.59667158e+00  2.59269953e+00  2.51119709e+00
  2.41686654e+00  2.34698176e+00  2.31308436e+00  2.31584001e+00
  2.34502411e+00  2.36918306e+00  2.34275222e+00  2.24031973e+00
  2.08990383e+00  1.96534204e+00  1.93312752e+00  1.99568236e+00
  2.08568144e+00  2.12022662e+00  2.06711268e+00  1.96341300e+00
  1.86766601e+00  1.78779888e+00  1.65169752e+00  1.35187984e+00
  8.35360527e-01  1.72481328e-01 -4.50105816e-01 -8.16307902e-01
 -7.99354017e-01 -4.40876782e-01  6.10874668e-02  4.54721302e-01
  5.71688831e-01  4.16962445e-01  1.53822973e-01 -8.39615799e-03
  4.7360591

In [ ]:
# train model over 40 epochs, using early stoppage if no improvement in average accuracy observed
def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for X_raw, X_bp, y in loader:
        X_raw = X_raw.to(device)
        X_bp  = X_bp.to(device)
        y     = y.to(device)

        optimizer.zero_grad()
        outputs = model(X_raw, X_bp)
        loss = criterion(outputs, y)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for X_raw, X_bp, y in loader:
            X_raw = X_raw.to(device)
            X_bp  = X_bp.to(device)

            logits = model(X_raw, X_bp)
            preds  = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(y.numpy())

    all_preds = np.array(all_preds)
    all_true  = np.array(all_true)

    overall_acc = (all_preds == all_true).mean()

    class_names = ["Interictal", "Preictal", "Ictal"]
    per_class = {}

    for i, name in enumerate(class_names):
        mask = (all_true == i)
        if mask.sum() > 0:
            per_class[name] = (all_preds[mask] == i).mean()
        else:
            per_class[name] = 0.0

    macro_acc = np.mean(list(per_class.values()))

    return overall_acc, macro_acc, per_class


# --- Training Loop ---
epochs = 40
patience = 15
counter = 0

best_macro_acc = 0
best_model_path = "best_eeg_model_one_channel.pth"


for epoch in range(epochs):

    train_loss = train_epoch(model, train_loader)

    train_acc, train_macro, train_per_class = evaluate(model, train_loader)
    test_acc,  test_macro,  test_per_class  = evaluate(model, test_loader)

    prev_lr = optimizer.param_groups[0]['lr']
    scheduler.step(test_macro)   # use balanced metric for LR scheduling
    new_lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {epoch+1} | Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}"
    )

    print(
        f"  Test per-class -> "
        f"Interictal: {test_per_class['Interictal']:.3f} | "
        f"Preictal: {test_per_class['Preictal']:.3f} | "
        f"Ictal: {test_per_class['Ictal']:.3f}"
    )

    print(f"  Macro Acc (balanced): {test_macro:.4f}")

    if new_lr != prev_lr:
        print(f"  --> LR reduced: {prev_lr:.2e} → {new_lr:.2e}")

    # --- Balanced model selection ---
    min_class_acc = min(test_per_class.values())

    if test_macro > best_macro_acc and min_class_acc > 0.40:
        best_macro_acc = test_macro
        counter = 0

        torch.save(model.state_dict(), best_model_path)

        full_path = os.path.abspath(best_model_path)
        print("  --> Model improved (balanced). Saved.")
        print(f"      Path: {full_path}")
        print(f"      Exists? {os.path.exists(full_path)}")

    else:
        counter += 1
        print(f"  --> No improvement. Patience: {counter}/{patience}")

    if counter >= patience:
        print("\nEarly Stopping Triggered!")
        break


print("\nTraining Complete. Loading best model...")
model.load_state_dict(torch.load(best_model_path))
print(f"Best Macro Accuracy: {best_macro_acc:.4f}")

In [ ]:
# evaluate trained model on held out test set
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

model.load_state_dict(torch.load('/content/drive/MyDrive/BME473PoP ML/final_model_4_8_26.pth'))
model.eval()
print("Loaded: final_model_4_8_26.pth")
all_preds = []
all_true = []
model.eval()
with torch.no_grad():
    for X_raw, X_bp, y in test_loader:
        X_raw = X_raw.to(device)
        X_bp  = X_bp.to(device)
        outputs = model(X_raw, X_bp)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y.numpy())

cm = confusion_matrix(all_true, all_preds)
class_names = ["Interictal", "Preictal", "Ictal"]

# Build custom annotation labels
descriptions = [
    ["Correct interictal\nclassifications",         "Misclassifications of\ninterictal windows\nas preictal",  "Misclassifications of\ninterictal windows\nas ictal"],
    ["Misclassifications of\npreictal windows\nas interictal", "Correct preictal\nclassifications",            "Misclassifications of\npreictal windows\nas ictal"],
    ["Misclassifications of\nictal windows\nas interictal",    "Misclassifications of\nictal windows\nas preictal", "Correct ictal\nclassifications"],
]

# Combine description + count into one annotation string
annot_labels = np.array([
    [f"{descriptions[i][j]}\n({cm[i][j]})" for j in range(3)]
    for i in range(3)
])

plt.figure(figsize=(12, 8))
sns.heatmap(cm, annot=annot_labels, fmt="", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names,
            linewidths=0.5,
            annot_kws={"size": 16})

plt.ylabel('True Label', fontsize=13)
plt.xlabel('Predicted Label', fontsize=13)
plt.title('Confusion Matrix Heatmap', fontsize=15)
plt.tight_layout()
plt.show()

report_dict = classification_report(all_true, all_preds,
                                    target_names=class_names,
                                    output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()
print("\n--- Detailed Performance Metrics ---")
try:
    display(df_report)
except NameError:
    print(df_report)

In [ ]:
# save best model state
import shutil
from google.colab import files

shutil.copy('/content/best_eeg_model_one_channel.pth', '/content/final_model_4_8_26.pth')
files.download('/content/final_model_4_8_26.pth')

In [ ]:
# direct testing of classifications on specific windows
class_names = ["Interictal", "Preictal", "Ictal"]

model.eval()
for class_idx, class_name in enumerate(class_names):
    # Find first window of this class in test set
    idx = np.where(np.array(all_true) == class_idx)[0][2]

    X_raw_sample = torch.tensor(X_test[idx:idx+1], dtype=torch.float32).to(device)
    X_bp_sample  = torch.tensor(X_test_bp[idx:idx+1], dtype=torch.float32).to(device)

    with torch.no_grad():
        logits = model(X_raw_sample, X_bp_sample)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]

    predicted = class_names[probs.argmax()]
    correct   = "✓" if predicted == class_name else "✗"

    print(f"True: {class_name} | Predicted: {predicted} {correct}")
    print(f"  Interictal: {probs[0]:.4f} | Preictal: {probs[1]:.4f} | Ictal: {probs[2]:.4f}")
    print()

In [ ]:
# save individual windows for classification testing on ESP32
for class_idx, class_name in enumerate(class_names):
    idx = np.where(np.array(all_true) == class_idx)[0][2]
    sample = X_test[idx, 0, :]  # shape: (1280,)

    filename = f"/content/eeg_{class_name.lower()}_sample_correct.h"
    with open(filename, "w") as f:
        f.write("#pragma once\n")
        f.write(f"// True label: {class_name}\n")
        f.write(f"const int EEG_SAMPLE_SIZE = {len(sample)};\n")
        f.write("const float EEG_SAMPLE_DATA[] = {\n  ")
        for i, val in enumerate(sample):
            f.write(f"{val:.6f}f")
            if i < len(sample) - 1:
                f.write(", ")
            if (i + 1) % 10 == 0:
                f.write("\n  ")
        f.write("\n};\n")

    print(f"Saved: {filename}")

# --- Download all three ---
from google.colab import files
files.download("/content/eeg_interictal_sample_correct.h")
files.download("/content/eeg_preictal_sample_correct.h")
files.download("/content/eeg_ictal_sample_correct.h")

In [ ]:
# save training artifacts of best model for hosting on web server
import json

training_artifacts = {
    "bp_mean": bp_mean.tolist(),
    "bp_std": bp_std.tolist(),
    "n_bp_features": int(n_bp_features)
}

with open("training_artifacts_final_model.json", "w") as f:
    json.dump(training_artifacts, f)
files.download("training_artifacts_final_model.json")